### Crop Classification and Species Mapping

| Code | Common Name | Scientific Name |
| :--- | :--- | :--- |
| BAR | Barley | *Hordeum vulgare* |
| BARS | Spring barley | *Hordeum vulgare* |
| BARW | Winter barley | *Hordeum vulgare* |
| WHB | Bread wheat | *Triticum aestivum* |
| WHBS | Spring bread wheat | *Triticum aestivum* |
| WHBW | Winter bread wheat | *Triticum aestivum* |
| WHD | Durum wheat | *Triticum turgidum durum* |
| WHDS | Spring durum wheat | *Triticum turgidum durum* |
| WHDW | Winter durum wheat | *Triticum turgidum durum* |
| MAZ | Maize | *Zea mays* |
| POT | Potato | *Solanum tuberosum* |
| RAP | Rapeseed | *Brassica napus* |
| RAPS | Spring rapeseed | *Brassica napus* |
| RAPW | Winter rapeseed | *Brassica napus* |
| SBT | Sugar beet | *Beta vulgaris* |
| SUN | Sunflower | *Helianthus annuus* |
| RYE | Rye | *Secale cereale* |
| RYES | Spring rye | *Secale cereale* |
| RYEW | Winter rye | *Secale cereale* |
| FAB | Faba bean | *Vicia faba* |

Tbase for GDD calculations
| Crop code | Tbase (°C) | Crop category |
|---|---:|---|
| BAR | 0.0 | Cereals |
| BARS | 0.0 | Cereals |
| BARW | 0.0 | Cereals |
| WHB | 0.0 | Cereals |
| WHBS | 0.0 | Cereals |
| WHBW | 0.0 | Cereals |
| WHD | 0.0 | Cereals |
| WHDS | 0.0 | Cereals |
| WHDW | 0.0 | Cereals |
| RYE | 0.0 | Cereals |
| RYES | 0.0 | Cereals |
| RYEW | 0.0 | Cereals |
| RAP | 0.0 | Oilseed rape |
| RAPS | 0.0 | Oilseed rape |
| RAPW | 0.0 | Oilseed rape |
| POT | 2.0 | Tubers & Roots |
| SBT | 3.0 | Tubers & Roots |
| SUGB | 3.0 | Tubers & Roots |
| FAB | 3.0 | Legumes |
| SUN | 5.0 | Warm-season crops |
| MAZ | 6.0 | Warm-season crops |

Imports

In [2]:
import sqlite3
from urllib.request import urlretrieve
from zipfile import ZipFile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import geopandas as gpd
import pandas as pd
import numpy as np
import cdsapi
import time

Python Dictionaries for crop name to crop code & crop code to Tbase (For GDD calculation)

In [3]:
# -------------------------------------------------------------
# 1. Crop Code to Common Name Mapping
# -------------------------------------------------------------
# Scientific Name -> List of Common Names
crop_lookup = {
    # Barley
    ("Hordeum vulgare", 1): ("BARS", "Spring barley"),
    ("Hordeum vulgare", 2): ("BARW", "Winter barley"),
    ("Hordeum vulgare", 0): ("BAR", "Barley"),
    # Bread wheat
    ("Triticum aestivum", 1): ("WHBS", "Spring bread wheat"),
    ("Triticum aestivum", 2): ("WHBW", "Winter bread wheat"),
    ("Triticum aestivum", 0): ("WHB", "Bread wheat"),
    # Durum wheat
    ("Triticum turgidum durum", 1): ("WHDS", "Spring durum wheat"),
    ("Triticum turgidum durum", 2): ("WHDW", "Winter durum wheat"),
    ("Triticum turgidum durum", 0): ("WHD", "Durum wheat"),
    # Rapeseed
    ("Brassica napus", 1): ("RAPS", "Spring rapeseed"),
    ("Brassica napus", 2): ("RAPW", "Winter rapeseed"),
    ("Brassica napus", 0): ("RAP", "Rapeseed"),
    # Rye
    ("Secale cereale", 1): ("RYES", "Spring rye"),
    ("Secale cereale", 2): ("RYEW", "Winter rye"),
    ("Secale cereale", 0): ("RYE", "Rye"),
    # Non-seasonal / Single-season crops
    ("Zea mays", 0): ("MAZ", "Maize"),
    ("Solanum tuberosum", 0): ("POT", "Potato"),
    ("Beta vulgaris", 0): ("SBT", "Sugar beet"),
    ("Helianthus annuus", 0): ("SUN", "Sunflower"),
    ("Vicia faba", 0): ("FAB", "Faba bean"),
}

# -------------------------------------------------------------
# 2. Crop Code to Base Temperature (T_base in °C) Mapping
# -------------------------------------------------------------
CROP_TBASE_MAP = {
    # Cereals (Tbase = 0.0 °C)
    "BAR": 0.0,
    "BARS": 0.0,
    "BARW": 0.0,
    "WHB": 0.0,
    "WHBS": 0.0,
    "WHBW": 0.0,
    "WHD": 0.0,
    "WHDS": 0.0,
    "WHDW": 0.0,
    "RYE": 0.0,
    "RYES": 0.0,
    "RYEW": 0.0,
    # Oilseed rape (Tbase = 0.0 °C)
    "RAP": 0.0,
    "RAPS": 0.0,
    "RAPW": 0.0,
    # Tubers & Roots
    "POT": 2.0,
    "SBT": 3.0,
    "SUGB": 3.0,
    # Legumes
    "FAB": 3.0,
    # Warm-season crops
    "SUN": 5.0,
    "MAZ": 6.0,
}

Change to UTF-8 from UTF-16 (Otherwise the file is not recognised) & ensure connection with the API client.

In [4]:
# Ensure .cdsapirc is UTF-8 encoded without BOM
config_path = Path.home() / ".cdsapirc"
raw = config_path.read_bytes()

if raw.startswith((b"\xff\xfe", b"\xfe\xff")):
    text = raw.decode("utf-16")
elif raw.startswith(b"\xef\xbb\xbf"):
    text = raw.decode("utf-8-sig")
else:
    text = raw.decode("utf-8")

config_path.write_text(text, encoding="utf-8")

client = cdsapi.Client()
print("CDS API client initialized successfully.")

2026-11-07 21:03:59,434 INFO [2026-10-06T00:00:00Z] **Data Stores Service protection measures** affecting data downloads and account access: [please read our Announcement](https://forum.ecmwf.int/t/data-stores-service-protection-measures-affecting-data-downloads-and-account-access-cds-ads-ewds-ecds/15481) before enquiring with ECMWF Support.


CDS API client initialized successfully.


Load the database & get the country from the unique coordinates of each observation 

In [5]:
# Load PEP725 dataset
pep_dataset = "pep725_data.csv"

# Load the dataset into a DataFrame
df_dataset = pd.read_csv(pep_dataset, sep=";")

# Keep relevant columns and rename
df_dataset = df_dataset[
    [
        "lon",
        "lat",
        "species",
        "phase_id",
        "year",
        "day",
        "date",
        "cult_season",
    ]
].rename(
    columns={
        "lon": "Longitude",
        "lat": "Latitude",
        "species": "CropName",
        "phase_id": "PhenoStageCode",
        "year": "Year",
        "day": "DOY",
        "date": "ObservationDate",
        "cult_season": "CultivationSeason",
    }
)

# Standardize inputs: trim whitespace and fill missing/non-seasonal values with 0
clean_species = df_dataset["CropName"].astype(str).str.strip()
clean_season = (
    pd.to_numeric(df_dataset["CultivationSeason"], errors="coerce")
    .fillna(0)
    .astype(int)
)

# Map scientific names to crop codes
# if cultivation season is 1 -> spring, if cultivation season is 2 -> winter, else -> not applicable
index_pairs = pd.MultiIndex.from_arrays([clean_species, clean_season])

# Map to tuple: (CropCode, CommonName)
mapped_records = index_pairs.map(crop_lookup)

# Extract into dedicated columns
df_dataset["CropCode"] = [
    r[0] if isinstance(r, tuple) else np.nan for r in mapped_records
]
df_dataset["CommonName"] = [
    r[1] if isinstance(r, tuple) else np.nan for r in mapped_records
]

# Parse dates and numeric stage codes
df_dataset["ObservationDate"] = pd.to_datetime(
    df_dataset["ObservationDate"],
    format="%d/%m/%Y",
    errors="coerce",
)
df_dataset["ObservationDate"] = pd.to_datetime(df_dataset["ObservationDate"], errors="coerce")
df_dataset["PhenoStageCode"] = pd.to_numeric(df_dataset["PhenoStageCode"], errors="coerce")
df_dataset["DOY"] = pd.to_numeric(df_dataset["DOY"], errors="coerce")


# Assign Country via Natural Earth Boundaries
unique_coords = df_dataset[["Latitude", "Longitude"]].drop_duplicates().copy()

unique_gdf = gpd.GeoDataFrame(
    unique_coords,
    geometry=gpd.points_from_xy(
        unique_coords["Longitude"], unique_coords["Latitude"]
    ),
    crs="EPSG:4326",
)

data_dir = Path("data")
shapefile = data_dir / "ne_110m_admin_0_countries.shp"

if not shapefile.exists():
    data_dir.mkdir(parents=True, exist_ok=True)
    zip_path = data_dir / "ne_110m_admin_0_countries.zip"
    urlretrieve(
        "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip",
        zip_path,
    )
    with ZipFile(zip_path) as archive:
        archive.extractall(data_dir)

countries = gpd.read_file(shapefile).to_crs("EPSG:4326")

country_lookup = gpd.sjoin(
    unique_gdf,
    countries[["ADMIN", "geometry"]],
    how="left",
    predicate="within",
)

country_lookup = (
    pd.DataFrame(country_lookup)[["Latitude", "Longitude", "ADMIN"]]
    .rename(columns={"ADMIN": "Country"})
    .drop_duplicates(subset=["Latitude", "Longitude"])
)

df_with_country = df_dataset.merge(
    country_lookup,
    on=["Latitude", "Longitude"],
    how="left",
    validate="many_to_one",
)

print(
    f"Loaded {len(df_with_country)} valid observations across {df_with_country['Country'].nunique()} countries."
)


C:\Users\jimpa\AppData\Local\Temp\ipykernel_28896\773453018.py:5: DtypeWarning: Columns (0: subspecies) have mixed types. Specify dtype option on import or set low_memory=False.
  df_dataset = pd.read_csv(pep_dataset, sep=";")


Loaded 129895 valid observations across 15 countries.


Extract selected country, selected crop & the corresponding dates of observations

In [6]:
# -------------------------------------------------------------
# 1. Select Country
# -------------------------------------------------------------
available_countries = sorted(df_with_country["Country"].dropna().unique())

print("Available countries:")
for idx, country in enumerate(available_countries, start=1):
    print(f"{idx}. {country}")

user_country_input = input("Enter the number or name of your chosen country: ").strip()

if user_country_input.isdigit() and 1 <= int(user_country_input) <= len(available_countries):
    selected_country = available_countries[int(user_country_input) - 1]
else:
    matched = [c for c in available_countries if c.lower() == user_country_input.lower()]
    if matched:
        selected_country = matched[0]
    else:
        raise ValueError(f"Country '{user_country_input}' not found in the list.")

print(f"\n--> Selected country: {selected_country}")

# -------------------------------------------------------------
# 2. Select Crop Type
# -------------------------------------------------------------
available_crops = sorted(
    df_with_country.loc[df_with_country["Country"] == selected_country, "CropCode"]
    .dropna()
    .unique()
)

print(f"\nAvailable crop types for {selected_country}:")
for idx, crop in enumerate(available_crops, start=1):
    print(f"{idx}. {crop}")

user_crop_input = input("Enter the number or code of your chosen crop type: ").strip()

if user_crop_input.isdigit() and 1 <= int(user_crop_input) <= len(available_crops):
    selected_crop = available_crops[int(user_crop_input) - 1]
else:
    matched_crop = [c for c in available_crops if c.lower() == user_crop_input.lower()]
    if matched_crop:
        selected_crop = matched_crop[0]
    else:
        raise ValueError(f"Crop type '{user_crop_input}' not found in the list.")

print(f"\n--> Selected crop type: {selected_crop}")

df_selected_country_crop = df_with_country[
    (df_with_country["Country"] == selected_country) &
    (df_with_country["CropCode"] == selected_crop)
].copy()

country_slug = selected_country.lower().replace(" ", "_")
crop_slug = selected_crop.lower().replace(" ", "_")

print(f"\nTotal records for {selected_country} - {selected_crop}: {len(df_selected_country_crop)}")


Available countries:
1. Austria
2. Belgium
3. Bosnia and Herzegovina
4. Croatia
5. Czechia
6. Germany
7. Hungary
8. Latvia
9. Montenegro
10. Netherlands
11. Poland
12. Slovakia
13. Spain
14. Sweden
15. Switzerland

--> Selected country: Germany

Available crop types for Germany:
1. BARW
2. MAZ
3. RYEW
4. SBT
5. SUN
6. WHBW

--> Selected crop type: BARW

Total records for Germany - BARW: 28886


Quality Control 

In [7]:
# 1. Enforce types with explicit Day-First format
df_selected_country_crop["PhenoStageCode"] = pd.to_numeric(
    df_selected_country_crop["PhenoStageCode"], errors="coerce"
)

initial_count = len(df_selected_country_crop)
print(f"Total raw observations:           {initial_count}")

# Drop true NaNs and exact duplicates
clean_obs = df_selected_country_crop.dropna(
    subset=["Latitude", "Longitude", "ObservationDate", "PhenoStageCode"]
).drop_duplicates(
    subset=["Latitude", "Longitude", "ObservationDate", "PhenoStageCode"]
)

# Allow valid range 0 to 100 (100 = Harvest in PEP725)
clean_obs = clean_obs[
    (clean_obs["PhenoStageCode"] >= 0) & (clean_obs["PhenoStageCode"] <= 100)
].copy()
print(f"Valid stages (0 to 100) retained: {len(clean_obs)}")

# -------------------------------------------------------------
# 2. Check Monotonicity
# -------------------------------------------------------------
clean_obs["Year"] = clean_obs["ObservationDate"].dt.year

# Sort by station, date, AND stage (handles multiple observations on same day)
clean_obs = clean_obs.sort_values(
    ["Latitude", "Longitude", "Year", "ObservationDate", "PhenoStageCode"]
).reset_index(drop=True)

# For winter crops: autumn (sowing/emergence) belongs to the upcoming harvest cycle.
# If evaluating within calendar year, allow the natural seasonal reset between harvest (>=80) and autumn sowing (<=10)
clean_obs["prev_stage"] = clean_obs.groupby(["Latitude", "Longitude", "Year"])[
    "PhenoStageCode"
].shift(1)

# A genuine error occurs when a stage regresses without being an autumn crop restart
is_seasonal_reset = (clean_obs["prev_stage"] >= 80) & (
    clean_obs["PhenoStageCode"] <= 10
)
clean_obs["stage_drop"] = (
    clean_obs["PhenoStageCode"] < clean_obs["prev_stage"]
) & (~is_seasonal_reset)

flagged_errors = clean_obs[clean_obs["stage_drop"]].copy()
print(f"True non-monotonic errors:        {len(flagged_errors)}")

clean_observations = (
    clean_obs[~clean_obs["stage_drop"]]
    .drop(columns=["prev_stage", "stage_drop"])
    .copy()
)
print(f"Final clean observations:         {len(clean_observations)}")

Total raw observations:           28886
Valid stages (0 to 100) retained: 28886
True non-monotonic errors:        91
Final clean observations:         28795


Select target BBCH 

In [8]:
# -------------------------------------------------------------
# Select target BBCH stage for DOY prediction
# -------------------------------------------------------------
stage_counts = (
    clean_observations["PhenoStageCode"]
    .value_counts()
    .sort_index()
    .reset_index()
)
stage_counts.columns = ["BBCH_Code", "Observation_Count"]

print(f"Recorded BBCH stages for {selected_country} - {selected_crop}:")
display(stage_counts.sort_values(by="Observation_Count", ascending=False).head(10).reset_index(drop=True))

user_target_bbch = input("\nEnter the target BBCH stage code (e.g., 99): ").strip()
target_bbch = float(user_target_bbch)

# Filter dataset strictly to the chosen target stage
df_target = clean_observations[clean_observations["PhenoStageCode"] == target_bbch].copy()

# Remove duplicate records for the same station and year if any exist
df_target = df_target.drop_duplicates(subset=["Latitude", "Longitude", "Year"]).reset_index(drop=True)

# Print first and last observation for the selected BBCH value
bbch_subset = clean_obs[clean_obs["PhenoStageCode"] == target_bbch]
first_date = bbch_subset["ObservationDate"].min()
last_date = bbch_subset["ObservationDate"].max()
print(f"First observation date for BBCH {int(target_bbch)}: {first_date.strftime('%B %d, %Y')}")
print(f"Last observation date for BBCH {int(target_bbch)}: {last_date.strftime('%B %d, %Y')}")
print(f"\n--> Selected Target BBCH: {int(target_bbch)}")
print(f"Total labeled observations available: {len(df_target)}")
print(f"Unique monitoring stations:           {df_target[['Latitude', 'Longitude']].drop_duplicates().shape[0]}")

Recorded BBCH stages for Germany - BARW:


,BBCH_Code,Observation_Count
0,100,5308
1,51,4961
2,85,4710
3,10,4660
4,0,4640
5,31,4516


First observation date for BBCH 51: April 20, 2017
Last observation date for BBCH 51: June 18, 2025

--> Selected Target BBCH: 51
Total labeled observations available: 4942
Unique monitoring stations:           839


Select Date Range & Aggregate to AgERA5 0.1° Grid

In [10]:
# -------------------------------------------------------------
# Select date range & aggregate coordinates
# -------------------------------------------------------------
date_input = input("Enter date range (e.g., 2017-01-01/2021-12-31): ").strip()

if "/" in date_input or " to " in date_input:
    sep = "/" if "/" in date_input else " to "
    start_raw, end_raw = date_input.split(sep, 1)
    start_date = pd.to_datetime(start_raw.strip()).strftime("%Y-%m-%d")
    end_date = pd.to_datetime(end_raw.strip()).strftime("%Y-%m-%d")
    date_query = f"{start_date}/{end_date}"
    tag_date = f"{start_date}_{end_date}"
else:
    raise ValueError("Please provide date range in 'YYYY-MM-DD/YYYY-MM-DD' format.")

print(f"\nFiltering target observations between {start_date} and {end_date}...")

# Filter target observations to the selected time window
location_dates_filtered = df_target[
    (df_target["ObservationDate"] >= start_date) & 
    (df_target["ObservationDate"] <= end_date)
].copy()

# Round to 0.1° AgERA5 grid
location_dates_filtered["AgERA5_Latitude"] = location_dates_filtered["Latitude"].round(1)
location_dates_filtered["AgERA5_Longitude"] = location_dates_filtered["Longitude"].round(1)

# Unique grid locations to query via CDS API
locations_to_query = (
    location_dates_filtered[["AgERA5_Longitude", "AgERA5_Latitude"]]
    .drop_duplicates()
    .rename(columns={"AgERA5_Longitude": "Longitude", "AgERA5_Latitude": "Latitude"})
    .reset_index(drop=True)
)

print(f"Target BBCH {int(target_bbch)} observations in period: {len(location_dates_filtered)}")
print(f"Unique AgERA5 0.1° grid cells to query:       {len(locations_to_query)}")

# Save filtered observations
output_dir = Path(f"output/{country_slug}_{crop_slug}_bbch{int(target_bbch)}_{tag_date}")
output_dir.mkdir(parents=True, exist_ok=True)
location_dates_filtered.to_csv(
    output_dir/f"AgERA5_all_observations.csv",
    sep=";",
    index=False
)

display(locations_to_query.head(10))


Filtering target observations between 2017-01-01 and 2025-12-31...
Target BBCH 51 observations in period: 4942
Unique AgERA5 0.1° grid cells to query:       782


,Longitude,Latitude
0,9.1,47.7
1,12.1,47.8
2,9.3,47.8
3,9.5,47.8
4,11.3,47.8
5,7.6,47.8
6,8.7,47.8
7,9.8,47.8
8,9.0,47.9
9,7.6,47.9


Request temperature data from the CDS 

In [11]:
dataset = "sis-agrometeorological-indicators-timeseries"

raw_dir = Path(f"data/agERA5_{country_slug}_{crop_slug}_bbch{int(target_bbch)}_{tag_date}_raw"
)
raw_dir.mkdir(parents=True, exist_ok=True)

variables = [
    "2m_temperature_24_hour_maximum",
    "2m_temperature_24_hour_mean",
    "2m_temperature_24_hour_minimum",
]

MAX_WORKERS = 2
MAX_ATTEMPTS = 5

# Filter out files that are already downloaded
pending_locations = []

for i, loc in locations_to_query.iterrows():
    lon = float(loc["Longitude"])
    lat = float(loc["Latitude"])
    filename = raw_dir / f"temperature_{i:04d}_{lat:.1f}_{lon:.1f}.csv"

    if not (filename.exists() and filename.stat().st_size > 500):
        pending_locations.append((i, lat, lon, filename))

print(
    f"Total locations: {len(locations_to_query)} | "
    f"Already downloaded: "
    f"{len(locations_to_query) - len(pending_locations)} | "
    f"Remaining: {len(pending_locations)}"
)


def download_location(item):
    idx, lat, lon, filename = item

    request = {
        "variable": variables,
        "location": {"longitude": lon, "latitude": lat},
        "date": [date_query],
        "data_format": "csv",
    }

    # Create a client per worker to avoid sharing client state
    client = cdsapi.Client()

    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            print(
                f"[{idx + 1}/{len(locations_to_query)}] "
                f"Requesting ({lat:.1f}N, {lon:.1f}E), "
                f"attempt {attempt}/{MAX_ATTEMPTS}"
            )

            client.retrieve(dataset, request, str(filename))

            # Optional: check the downloaded file
            if not filename.exists() or filename.stat().st_size == 0:
                raise RuntimeError("Downloaded file is missing or empty")

            time.sleep(2)
            return idx, True, None

        except Exception as e:
            error_msg = str(e).lower()

            if attempt == MAX_ATTEMPTS:
                return idx, False, str(e)

            if (
                "temporarily limited" in error_msg
                or "400" in error_msg
                or "queue" in error_msg
                or "429" in error_msg
            ):
                wait_seconds = 60
                print(
                    f"[{idx + 1}] CDS queue/rate limit; "
                    f"retrying in {wait_seconds}s: {e}"
                )
            else:
                wait_seconds = 15
                print(
                    f"[{idx + 1}] Error: {e}. "
                    f"Retrying in {wait_seconds}s..."
                )

            time.sleep(wait_seconds)

    return idx, False, "Maximum attempts exceeded"


failed = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [
        executor.submit(download_location, item)
        for item in pending_locations
    ]

    for future in as_completed(futures):
        idx, success, error = future.result()

        if success:
            print(f"[{idx + 1}] Download completed.")
        else:
            failed.append((idx, error))
            print(f"[{idx + 1}] FAILED: {error}")

print("\nProcessing completed.")
print(f"Successful downloads: {len(pending_locations) - len(failed)}")
print(f"Failed downloads: {len(failed)}")
print(f"Output directory: {raw_dir}")

Total locations: 782 | Already downloaded: 0 | Remaining: 782


2026-11-07 21:05:46,331 INFO [2026-10-06T00:00:00Z] **Data Stores Service protection measures** affecting data downloads and account access: [please read our Announcement](https://forum.ecmwf.int/t/data-stores-service-protection-measures-affecting-data-downloads-and-account-access-cds-ads-ewds-ecds/15481) before enquiring with ECMWF Support.
2026-11-07 21:05:46,373 INFO [2026-10-06T00:00:00Z] **Data Stores Service protection measures** affecting data downloads and account access: [please read our Announcement](https://forum.ecmwf.int/t/data-stores-service-protection-measures-affecting-data-downloads-and-account-access-cds-ads-ewds-ecds/15481) before enquiring with ECMWF Support.


[2/782] Requesting (47.8N, 12.1E), attempt 1/5
[1/782] Requesting (47.7N, 9.1E), attempt 1/5


2026-11-07 21:05:46,636 INFO Request ID is 63882050-433a-4e41-ba1f-a2c1180db6e2
2026-11-07 21:05:46,651 INFO Request ID is b6757462-233e-4122-ad50-4f7d3e43056e
2026-11-07 21:05:46,729 INFO status has been updated to successful
2026-11-07 21:05:46,742 INFO status has been updated to successful


86c97d5930e569bd40e621f6ed0b3971.csv:   0%|          | 0.00/241k [00:00<?, ?B/s]

b1ad414b6ef326be55b4f8c4571ea25f.csv:   0%|          | 0.00/247k [00:00<?, ?B/s]

... (3,901 outputs left out)

35f5803cee52bdc75954d8295e103611.csv:   0%|          | 0.00/238k [00:00<?, ?B/s]

2026-11-08 00:46:25,161 INFO status has been updated to successful


8d6012d6b565d391e9fc5931a83dd8fe.csv:   0%|          | 0.00/238k [00:00<?, ?B/s]

[781] Download completed.
[782] Download completed.

Processing completed.
Successful downloads: 782
Failed downloads: 0
Output directory: data\agERA5_germany_barw_bbch51_2017-01-01_2025-12-31_raw


Combine all different CSVs for the different locations into one

In [12]:
# -------------------------------------------------------------
# Combine raw weather CSV files (or load combined file if raw files were already deleted)
# -------------------------------------------------------------
combined_weather_file = (
    output_dir
    / f"AgERA5_observations_combined_with_weather_data.csv"
)

raw_files = sorted(raw_dir.glob("temperature_*.csv")) if raw_dir.exists() else []

if raw_files:
    weather_parts = [pd.read_csv(f) for f in raw_files]
    weather_df = pd.concat(weather_parts, ignore_index=True)

    weather_df["valid_time"] = pd.to_datetime(
        weather_df["valid_time"], errors="coerce"
    )

    weather_df = weather_df.rename(
        columns={
            "Temperature_Air_2m_Max_24h": "Tmax_C",
            "Temperature_Air_2m_Mean_24h": "Tmean_C",
            "Temperature_Air_2m_Min_24h": "Tmin_C",
            "latitude": "AgERA5_Latitude",
            "longitude": "AgERA5_Longitude",
        }
    )

    # Convert Kelvin to Celsius
    for col in ["Tmax_C", "Tmean_C", "Tmin_C"]:
        weather_df[col] = pd.to_numeric(weather_df[col], errors="coerce") - 273.15

    # Round grid coordinates to 1 decimal place to ensure exact merge matching
    weather_df["AgERA5_Latitude"] = weather_df["AgERA5_Latitude"].round(1)
    weather_df["AgERA5_Longitude"] = weather_df["AgERA5_Longitude"].round(1)

    # Drop any duplicate weather rows
    weather_df = (
        weather_df.drop_duplicates(
            subset=["valid_time", "AgERA5_Latitude", "AgERA5_Longitude"]
        )
        .sort_values(["AgERA5_Latitude", "AgERA5_Longitude", "valid_time"])
        .reset_index(drop=True)
    )

    # Save combined weather CSV

    weather_df.to_csv(combined_weather_file, sep=";", index=False, encoding="utf-8")
    print(f"Combined {len(raw_files)} raw files into: {combined_weather_file.resolve()}")

elif combined_weather_file.exists():
    print(f"Raw files were already cleaned up. Loading existing combined file: {combined_weather_file.resolve()}")
    weather_df = pd.read_csv(combined_weather_file, sep=";")
    weather_df["valid_time"] = pd.to_datetime(
        weather_df["valid_time"], errors="coerce"
    )
    weather_df["AgERA5_Latitude"] = weather_df["AgERA5_Latitude"].round(1)
    weather_df["AgERA5_Longitude"] = weather_df["AgERA5_Longitude"].round(1)

else:
    raise FileNotFoundError(
        f"Neither raw files in {raw_dir} nor {combined_weather_file} were found."
    )

print(f"Weather dataset ready. Total daily records: {len(weather_df)}")
display(weather_df.head())

Combined 782 raw files into: C:\Users\jimpa\Desktop\Thesis\Notebooks\output\germany_barw_bbch51_2017-01-01_2025-12-31\AgERA5_observations_combined_with_weather_data.csv
Weather dataset ready. Total daily records: 2570434


,valid_time,Tmax_C,Tmean_C,Tmin_C,AgERA5_Latitude,AgERA5_Longitude
0,2017-01-01,0.98974,-2.36030,-4.10930,47.7,9.1
1,2017-01-02,1.49307,-1.47587,-4.36777,47.7,9.1
2,2017-01-03,-0.09543,-2.53947,-4.29710,47.7,9.1
3,2017-01-04,1.06747,-0.73487,-2.88990,47.7,9.1
4,2017-01-05,0.86724,-2.71247,-5.13320,47.7,9.1


Calculate the GDD baseline 

In [13]:
# -------------------------------------------------------------
# 1. Base temperature, Daily GDD, and Accumulated GDD (AGDD)
# -------------------------------------------------------------
T_BASE = CROP_TBASE_MAP.get(selected_crop, 0.0)
weather_df["daily_gdd"] = (weather_df["Tmean_C"] - T_BASE).clip(lower=0.0)

# Sort chronologically and reset Accumulated GDD (AGDD) on January 1 of each year
weather_df["Year"] = weather_df["valid_time"].dt.year
weather_df["Weather_DOY"] = weather_df["valid_time"].dt.dayofyear
weather_df = weather_df.sort_values(
    ["AgERA5_Latitude", "AgERA5_Longitude", "valid_time"]
)

weather_df["agdd"] = weather_df.groupby(
    ["AgERA5_Latitude", "AgERA5_Longitude", "Year"]
)["daily_gdd"].cumsum()

# Restrict to Days 1 to 365 for consistent column dimensions across all years
weather_365 = weather_df[weather_df["Weather_DOY"] <= 365].copy()

# -------------------------------------------------------------
# 2. Pivot Daily GDD (GDD_doy_1 ... GDD_doy_365)
# -------------------------------------------------------------
gdd_wide = weather_365.pivot_table(
    index=["AgERA5_Latitude", "AgERA5_Longitude", "Year"],
    columns="Weather_DOY",
    values="daily_gdd",
)
gdd_feature_cols = [f"GDD_doy_{int(col)}" for col in gdd_wide.columns]
gdd_wide.columns = gdd_feature_cols
gdd_wide = gdd_wide.reset_index()

# -------------------------------------------------------------
# 3. Pivot Accumulated GDD (AGDD_doy_1 ... AGDD_doy_365)
# -------------------------------------------------------------
agdd_wide = weather_365.pivot_table(
    index=["AgERA5_Latitude", "AgERA5_Longitude", "Year"],
    columns="Weather_DOY",
    values="agdd",
)
agdd_feature_cols = [f"AGDD_doy_{int(col)}" for col in agdd_wide.columns]
agdd_wide.columns = agdd_feature_cols
agdd_wide = agdd_wide.reset_index()

# -------------------------------------------------------------
# 4. Prepare target DOY and merge both GDD and AGDD profiles
# -------------------------------------------------------------
location_dates_filtered["ObservationDate"] = pd.to_datetime(
    location_dates_filtered["ObservationDate"]
)
location_dates_filtered["Year"] = location_dates_filtered["ObservationDate"].dt.year
location_dates_filtered["DOY"] = location_dates_filtered["ObservationDate"].dt.dayofyear

# Keep earliest onset date per station per year
location_dates_filtered = (
    location_dates_filtered.sort_values(
        ["Latitude", "Longitude", "Year", "ObservationDate"]
    )
    .drop_duplicates(subset=["Latitude", "Longitude", "Year"], keep="first")
    .reset_index(drop=True)
)

# Merge Daily GDD and Accumulated GDD strictly by Location and Year (Zero Leakage)
df_final = location_dates_filtered.merge(
    gdd_wide,
    on=["AgERA5_Latitude", "AgERA5_Longitude", "Year"],
    how="left",
).merge(
    agdd_wide,
    on=["AgERA5_Latitude", "AgERA5_Longitude", "Year"],
    how="left",
)

# -------------------------------------------------------------
# 5. Save the single master modeling dataset
# -------------------------------------------------------------
final_output_file = (
    output_dir
    / f"curated_for_modeling.csv"
)
df_final.to_csv(final_output_file, sep=";", index=False, encoding="utf-8")

print(f"\nMaster modeling dataset saved to: {final_output_file.resolve()}")
print(f"Total labeled rows:               {len(df_final)}")
print(f"Daily GDD columns added:          {len(gdd_feature_cols)} (GDD_doy_1 to GDD_doy_365)")
print(f"Accumulated GDD columns added:    {len(agdd_feature_cols)} (AGDD_doy_1 to AGDD_doy_365)")
print(f"Missing weather rows:             {df_final['AGDD_doy_180'].isna().sum()}")

display(
    df_final[
        [
            "Latitude",
            "Longitude",
            "Year",
            "DOY",
            "GDD_doy_150",
            "GDD_doy_180",
            "AGDD_doy_150",
            "AGDD_doy_180",
        ]
    ].head(10)
)


Master modeling dataset saved to: C:\Users\jimpa\Desktop\Thesis\Notebooks\output\germany_barw_bbch51_2017-01-01_2025-12-31\curated_for_modeling.csv
Total labeled rows:               4942
Daily GDD columns added:          365 (GDD_doy_1 to GDD_doy_365)
Accumulated GDD columns added:    365 (AGDD_doy_1 to AGDD_doy_365)
Missing weather rows:             0


,Latitude,Longitude,Year,DOY,GDD_doy_150,GDD_doy_180,AGDD_doy_150,AGDD_doy_180
0,47.7333,9.06667,2017,140,21.29693,16.81268,990.36996,1594.61623
1,47.7333,9.06667,2018,125,20.22490,19.43023,1079.54076,1638.11766
2,47.7333,9.06667,2019,138,14.21250,23.10200,899.66900,1488.29911
3,47.7333,9.06667,2020,132,14.33755,20.08242,1084.93818,1590.74364
4,47.7333,9.06667,2021,142,13.89086,17.45590,822.05834,1399.22643
5,47.7500,12.11670,2017,134,20.03542,15.29626,878.47218,1420.17207
6,47.7500,12.11670,2018,122,19.70425,15.54232,997.38514,1495.87614
7,47.7500,12.11670,2019,127,11.21575,20.98364,794.45081,1367.20886
8,47.7500,12.11670,2020,123,9.69216,20.03732,930.49384,1375.71660
9,47.7500,12.11670,2021,136,10.69528,17.69048,676.12137,1216.52486
